# Reproducing EPA's Lead and Copper Rule Improvements Analysis from Its Published Tables — a Reproduction That Misses

**Artifact class:** `tutorial`  ·  **Data mode:** `empirical`  ·  **Source:** U.S. EPA, *Economic Analysis for the Final Lead and Copper Rule Improvements* (October 2024), Exhibits 6-4, 6-5 and 6-6

This notebook uses **real data only**: values EPA published, committed as tables with their exhibit and PDF page. It asks one question: **can EPA's own printed annualised costs and benefits at a 2 percent discount rate be rebuilt from the yearly streams EPA also prints, under a documented timing convention?** It was planned as a cost-benefit re-analysis. The first reproduction check showed the answer is no, and the plan's own stop rule then applies: the notebook reports that finding and builds nothing on it. **No result at 3 or 7 percent and no net present value at any rate is presented here.**

This is a reproduction attempt of EPA's own assumptions, **not an independent estimate**. A failed reproduction does not show EPA's figures are wrong; it shows they cannot be rebuilt from the published tables under a documented convention.

*Proprietary. SPDX-License-Identifier: LicenseRef-Proprietary*

## Learning Objectives

By the end of this notebook you will be able to:

1. Rebuild an annualised value from a yearly stream, and say which parts of the convention (base year, timing within a year, annuity form) the result does and does not depend on.
2. State a reproduction tolerance in advance and report a miss as a finding with its size.
3. Read a regulatory impact analysis for what it states about discounting, and for what it leaves unstated.
4. Keep a written record of a plan, a check, an amendment and a result so that a reader can see which came first.

## Prerequisites

**Knowledge.** Present value and discount factors; the difference between an annualised value and an undiscounted yearly flow; what a pre-analysis plan fixes. **Software.** None beyond the base installation: the notebook reads committed tables and does not need the source PDF. **Data access.** None.

## Data Provenance

| Item | Meaning |
|---|---|
| `epa_lcri_yearly_streams.csv` | EPA's yearly undiscounted incremental costs, benefits and net benefits, 2024 to 2058 (35 years), millions of 2022 dollars, for EPA's **low** and **high** scenarios (Exhibit 6-4, Exhibit 6-5) |
| `epa_lcri_printed_annualized_2pct.csv` | EPA's printed annualised incremental costs, benefits and net benefits at 2 percent, low and high (Exhibit 6-6, also Exhibit ES-3) |

Source document: one file, `508_lcri_final_ea_10-21-2024.pdf`, 9,589,288 bytes, `Last-Modified` 24 October 2024, SHA-256 `057c531f5bf14910f321a653bcdf341317676325b0d3dafeef4e874054bca350`, downloaded once on 2026-10-30 after the owner approved exactly that file, and cached outside the repository (it is never committed). The tables were extracted by `scripts/extract_epa_lcri_tables.py`, which stops unless the file has that digest; each row carries its exhibit and PDF page, and `data/SOURCES.md` records the digests of both tables. The retrieval record and run ledger are in `provenance/`, bound to the plan.

## The sequence, in order

1. **The plan** (the file E02_pap.md in the notebook's pap folder, committed alone on 2026-10-30 **before** the document was downloaded) fixed the headline quantities, the discount rates, the reproduction check with its tolerance, the reporting of EPA's low and high estimates, the checks and the stop rules. It assumed a central scenario and left the timing convention to be read from the document.
2. **The retrieval.** The document was downloaded once; its size matched the approved size before the body was fetched.
3. **The first check.** The streams were located (the plan's first stop rule does not fire: EPA publishes yearly streams). A first reproduction, under the plan's reading of discounting to year one with annualisation by the same factors, missed EPA's printed values by 1.8 to 2.0 percent. The plan's stop rule for a missed reproduction fired and the owner was asked.
4. **The owner's timing rule**, committed as **amendment 1** (the file E02_pap_amendment_1.md in the pap folder, relation `after_retrieval`): use the convention EPA's analysis states; where it states none, use Circular A-4 (2003); apply it once and try no alternative; and replace the plan's central scenario with EPA's two published scenarios, paired as EPA publishes them. The amendment states that the gap of about a factor of 1.02 had been seen when the rule was set.
5. **The single application.** The convention is applied once, below.
6. **The miss.** All six comparisons miss the plan's tolerance. The stop rule holds. The owner then directed that the notebook be built with the gap as its finding and with nothing computed at other rates.

In [1]:
"""Configuration. Every analytic choice is named here, not inferred."""
from __future__ import annotations

import hashlib
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        REPO_ROOT = _cand
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.methods import discount

NB_DIR = REPO_ROOT / "projects" / "10-economic-evaluation" / "E02_cost_benefit_reanalysis"
DATA = NB_DIR / "data"
PLAN, AMEND = NB_DIR / "pap" / "E02_pap.md", NB_DIR / "pap" / "E02_pap_amendment_1.md"
RATE = 0.02                      # the rate EPA used (document section 4.2.3.3, page 4-13); the only rate computed here
REL_TOL, ABS_TOL = 0.005, 0.1    # plan section 5: the larger of 0.5 percent and one unit in the last printed digit (one tenth of a million dollars)
TARGET = "net_benefits_musd"
SCENARIOS = ("low", "high")      # EPA's two published scenarios, costs paired with benefits (amendment 1)
EXPECTED_SOURCE_SHA = "057c531f5bf14910f321a653bcdf341317676325b0d3dafeef4e874054bca350"


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


print(f"plan       : {PLAN.relative_to(REPO_ROOT)}\n  SHA-256  : {sha256(PLAN)}")
print(f"amendment 1: {AMEND.relative_to(REPO_ROOT)}\n  SHA-256  : {sha256(AMEND)}")
print(f"discount rate used here: {RATE:.0%}; tolerance: the larger of {REL_TOL:.1%} of EPA's printed value and {ABS_TOL} (million 2022 dollars)")
print("no random-number call, no other rate, and no present value is printed in this notebook")

plan       : projects/10-economic-evaluation/E02_cost_benefit_reanalysis/pap/E02_pap.md
  SHA-256  : 91e02c694a3ba14f1e9fc184477c00384ae4a52479aceaff15d56510a6f01f74
amendment 1: projects/10-economic-evaluation/E02_cost_benefit_reanalysis/pap/E02_pap_amendment_1.md
  SHA-256  : 7db39861f47cc89ee3ce59d06a7f38efdb92e4bb53aa1281a2d5e8f7d0d5e78c
discount rate used here: 2%; tolerance: the larger of 0.5% of EPA's printed value and 0.1 (million 2022 dollars)
no random-number call, no other rate, and no present value is printed in this notebook


## The rule's regulatory status

EPA published the final rule in the Federal Register on 30 October 2024 (document 2024-23549). **As of 31 October 2026**, EPA's program page for the rule, whose own date line reads "Last updated on December 29, 2025," shows the rule as final and mentions no stay, delay, reconsideration, rescission or litigation. It describes a general timeline in which systems identify and replace lead pipes within 10 years. EPA's Federal Register notice sets compliance three years after promulgation, with a state or EPA extension of the compliance start for treatment-technique requirements possible for up to five years. **Court dockets were not searched.** Re-analysing a published analysis does not depend on this status: the question here is what EPA's published tables imply, whether or not the rule is later changed, delayed or withdrawn.

## 1. The tables are the ones the record names

Before anything is computed, the two tables are checked against the digests recorded in `data/SOURCES.md`, and each scenario's yearly rows are checked for completeness and for internal consistency (net equals benefits minus costs, within EPA's rounding).

In [2]:
sources_md = (DATA / "SOURCES.md").read_text(encoding="utf-8")
ok_digests = []
for name in ("epa_lcri_yearly_streams.csv", "epa_lcri_printed_annualized_2pct.csv"):
    recorded = re.search(re.escape(name) + r"`\s*\|\s*`([0-9a-f]{64})`", sources_md).group(1)
    ok_digests.append(sha256(DATA / name) == recorded)
    print(f"{name}: SHA-256 {sha256(DATA / name)[:16]}..., recorded in SOURCES.md: {ok_digests[-1]}")
src_recorded = re.search(r"SHA-256 \| `([0-9a-f]{64})`", sources_md).group(1)
print(f"source document digest recorded in SOURCES.md equals the one in this notebook: {src_recorded == EXPECTED_SOURCE_SHA}")

streams = pd.read_csv(DATA / "epa_lcri_yearly_streams.csv")
printed = pd.read_csv(DATA / "epa_lcri_printed_annualized_2pct.csv")
assert streams[TARGET].var() > 0, "the yearly net benefits must vary"
years_ok = all(sorted(streams.loc[streams.scenario == s, "year"]) == list(range(2024, 2059)) for s in SCENARIOS)
net_gap = ((streams["incremental_benefits_musd"] - streams["incremental_costs_musd"]) - streams[TARGET]).abs().max()
totals = streams.groupby("scenario")[TARGET].sum() / 1000.0
print(f"\n35 yearly rows per scenario, 2024 to 2058: {years_ok}; largest |net - (benefits - costs)| {net_gap:.2f} million")
print(f"undiscounted net benefits: low {totals['low']:,.1f} billion, high {totals['high']:,.1f} billion "
      f"(EPA's text, Chapter 6: 'range from $479 billion to $918 billion')")
tables_ok = all(ok_digests) and src_recorded == EXPECTED_SOURCE_SHA and years_ok and net_gap <= 0.1001 and round(totals["low"]) == 479 and round(totals["high"]) == 918
print(f"FALSIFICATION: tables_match_record statistic={net_gap:.3f} largest net-versus-(benefits-minus-costs) gap in millions; digests, years and EPA's printed undiscounted totals all agree verdict={'PASS' if tables_ok else 'FAIL'}")

epa_lcri_yearly_streams.csv: SHA-256 b1e361af472cf207..., recorded in SOURCES.md: True
epa_lcri_printed_annualized_2pct.csv: SHA-256 521ce3c50560b9b0..., recorded in SOURCES.md: True
source document digest recorded in SOURCES.md equals the one in this notebook: True

35 yearly rows per scenario, 2024 to 2058: True; largest |net - (benefits - costs)| 0.10 million
undiscounted net benefits: low 479.2 billion, high 918.5 billion (EPA's text, Chapter 6: 'range from $479 billion to $918 billion')
FALSIFICATION: tables_match_record statistic=0.100 largest net-versus-(benefits-minus-costs) gap in millions; digests, years and EPA's printed undiscounted totals all agree verdict=PASS


## 2. What EPA published at 2 percent

EPA's printed annualised incremental costs, benefits and net benefits at a 2 percent discount rate, in millions of 2022 dollars, for its two scenarios (Exhibit 6-6, repeated in Exhibit ES-3). **Costs and benefits are paired within each scenario exactly as EPA publishes them**: the low costs go with the low benefits. There is no central scenario, midpoint or constructed scenario.

In [3]:
published = printed.pivot(index="quantity", columns="scenario", values="value_musd").loc[
    ["annualized_incremental_costs", "annualized_incremental_benefits", "annual_net_benefits"], list(SCENARIOS)]
print("EPA's printed annualised incremental values at 2 percent (millions of 2022 dollars)")
print(published.round(1).to_string())
print(f"\nsource: {printed['source_exhibit'].iloc[0]}, PDF page {printed['pdf_page'].iloc[0]}")

EPA's printed annualised incremental values at 2 percent (millions of 2022 dollars)
scenario                             low     high
quantity                                         
annualized_incremental_costs      1468.8   1953.8
annualized_incremental_benefits  13493.5  25142.6
annual_net_benefits              12024.7  23188.8

source: Exhibit 6-6, PDF page 711


## 3. What EPA states about discounting, and what it leaves unstated

EPA's analysis states: the rate ("the EPA discounted future costs (and benefits) at a 2 percent discount rate," Section 4.2.3.3, "Discounting and Cost of Capital," document page 4-13); the base year ("Benefits are discounted back to year one of the analysis and annualized within the SafeWater LCR model," Section 5.6.2, "Monetized National Annual Benefits," document page 5-58); and that costs and benefits are annualised over the same 35-year period "symmetrically" (Executive Summary, document page ES-2). Section 4.2.3.3 also says that, for impacts on water systems and households, EPA "uses the estimated PWS cost of capital to discount future costs" (document page 4-14); the national costs are printed under the heading of a 2 percent discount rate, and this notebook takes them as such.

**EPA's analysis states no convention** for whether flows fall at the start or the end of each year, and none for the annuity form used to annualise. A search of the whole document for those terms found none. The plan therefore turns, as the owner's rule says, to **Circular A-4, September 17, 2003**, Section "Discount Rates," subsection 1 "The Rationale for Discounting," page 32: the discount factor "is 1/(1+ the discount rate)^t where t measures the number of years in the future that the benefits or costs are expected to occur." The same Circular, in its section on presenting results (page 45), says the "stream of annualized estimates should begin in the year in which the final rule will begin to have effects." It states no start-of-year or end-of-year convention either.

**The convention, applied once.** The base year is year one, 2024, so a flow in year *y* is discounted by 1/(1+r)^(y−2024) and year one is undiscounted. The annualised value is the constant yearly amount over the same 35 years that has the same present value under the same factors. Because it divides a present value by the sum of the same factors, **the annualised value does not depend on whether flows are placed at the start or end of a year**; only the level of a present value would, and none is shown here.

## 4. The single application, and the six comparisons

For each scenario and each of the three quantities: annualise EPA's yearly stream at 2 percent under the convention above, and compare with EPA's printed value. The tolerance was fixed in the plan before the document was read: **the larger of 0.5 percent of EPA's printed value and one unit in the last printed digit**.

In [4]:
rows = []
for s in SCENARIOS:
    x = streams[streams.scenario == s].sort_values("year")
    for quantity, column in (("annualized_incremental_costs", "incremental_costs_musd"),
                             ("annualized_incremental_benefits", "incremental_benefits_musd"),
                             ("annual_net_benefits", "net_benefits_musd")):
        flows = x[column].to_numpy(dtype=float)
        # Present value of the stream with year one undiscounted, divided by the sum of the same factors (computed, never printed).
        rebuilt = discount.present_value(flows, RATE, timing="start") / discount.annuity_factor(RATE, len(flows), timing="start")
        epa = float(published.loc[quantity, s])
        gap = rebuilt - epa
        tol = max(REL_TOL * abs(epa), ABS_TOL)
        rows.append({"scenario": s, "quantity": quantity, "rebuilt": rebuilt, "EPA printed": epa, "gap": gap,
                     "gap %": 100 * gap / epa, "tolerance": tol, "within": abs(gap) <= tol, "EPA / rebuilt": epa / rebuilt})
compare = pd.DataFrame(rows)
for _, r in compare.iterrows():
    print(f"{r.scenario:4s} {r.quantity:34s} rebuilt {r.rebuilt:10.1f}  EPA {r['EPA printed']:10.1f}  gap {r.gap:8.1f} ({r['gap %']:+.2f}%)  "
          f"tolerance {r.tolerance:6.1f}  {'within' if r.within else 'OUTSIDE'}")
n_out = int((~compare["within"]).sum())
print(f"\n{n_out} of {len(compare)} comparisons are outside the plan's tolerance; gaps run from {compare['gap %'].abs().min():.2f}% to {compare['gap %'].abs().max():.2f}% in absolute size")
print(f"FALSIFICATION: reproduction_at_epa_rate statistic={compare['gap %'].abs().max():.2f} largest absolute gap in percent over six comparisons; plan tolerance {100 * REL_TOL:.1f} percent verdict={'PASS' if n_out == 0 else 'FAIL'}")

low  annualized_incremental_costs       rebuilt     1442.3  EPA     1468.8  gap    -26.5 (-1.80%)  tolerance    7.3  OUTSIDE
low  annualized_incremental_benefits    rebuilt    13228.9  EPA    13493.5  gap   -264.6 (-1.96%)  tolerance   67.5  OUTSIDE
low  annual_net_benefits                rebuilt    11786.6  EPA    12024.7  gap   -238.1 (-1.98%)  tolerance   60.1  OUTSIDE
high annualized_incremental_costs       rebuilt     1919.3  EPA     1953.8  gap    -34.5 (-1.77%)  tolerance    9.8  OUTSIDE
high annualized_incremental_benefits    rebuilt    24649.7  EPA    25142.6  gap   -492.9 (-1.96%)  tolerance  125.7  OUTSIDE
high annual_net_benefits                rebuilt    22730.4  EPA    23188.8  gap   -458.4 (-1.98%)  tolerance  115.9  OUTSIDE

6 of 6 comparisons are outside the plan's tolerance; gaps run from 1.77% to 1.98% in absolute size
FALSIFICATION: reproduction_at_epa_rate statistic=1.98 largest absolute gap in percent over six comparisons; plan tolerance 0.5 percent verdict=FAIL


### What the gap looks like, and what it does not tell us

The rebuilt values are all below EPA's, by nearly the same proportion. The next cell prints the ratio of EPA's value to the rebuilt value for each comparison, beside one year of discounting at 2 percent.

In [5]:
one_year = 1.0 + RATE
print(f"one year of discounting at {RATE:.0%}: factor {one_year:.4f}")
for _, r in compare.iterrows():
    print(f"{r.scenario:4s} {r.quantity:34s} EPA / rebuilt = {r['EPA / rebuilt']:.4f}  (difference from one year of discounting {r['EPA / rebuilt'] - one_year:+.4f})")
spread = compare["EPA / rebuilt"]
print(f"\nratios range from {spread.min():.4f} to {spread.max():.4f}")
# A check of the library on a stream with a known answer: a constant stream annualises to itself at the rate EPA used.
const = discount.present_value(np.full(35, 100.0), RATE, timing="start") / discount.annuity_factor(RATE, 35, timing="start")
print(f"FALSIFICATION: annualisation_identity statistic={abs(const - 100.0):.2e} absolute error annualising a constant stream of 100 over 35 years verdict={'PASS' if abs(const - 100.0) < 1e-9 else 'FAIL'}")

one year of discounting at 2%: factor 1.0200
low  annualized_incremental_costs       EPA / rebuilt = 1.0184  (difference from one year of discounting -0.0016)
low  annualized_incremental_benefits    EPA / rebuilt = 1.0200  (difference from one year of discounting -0.0000)
low  annual_net_benefits                EPA / rebuilt = 1.0202  (difference from one year of discounting +0.0002)
high annualized_incremental_costs       EPA / rebuilt = 1.0180  (difference from one year of discounting -0.0020)
high annualized_incremental_benefits    EPA / rebuilt = 1.0200  (difference from one year of discounting -0.0000)
high annual_net_benefits                EPA / rebuilt = 1.0202  (difference from one year of discounting +0.0002)

ratios range from 1.0180 to 1.0202
FALSIFICATION: annualisation_identity statistic=8.53e-14 absolute error annualising a constant stream of 100 over 35 years verdict=PASS


A gap of this size is **consistent with a difference in the base year or in the timing of flows** between EPA's model and the convention applied here: the ratios sit close to one year of discounting. It is only consistent with it. **EPA's analysis states no convention, so the cause cannot be confirmed from the published tables.** Other differences are possible: Section 4.2.3.3 says costs for water systems and households are discounted at the systems' cost of capital, and the SafeWater LCR model works from its own unrounded, system-level flows, of which the yearly national table is a sum. No alternative convention was tried here, not even as a labelled sensitivity; the owner's rule was to apply the stated convention once.

## 5. The stop rule holds

The plan fixed that, if the reproduction at EPA's own rate misses its tolerance, the notebook **does not go on to present re-discounted figures as comparable with EPA's headline.** Accordingly, this notebook presents **no result at 3 percent or 7 percent** (the plan's primary rates under the 2003 Circular), **no net present value at any rate, no switching discount rate and no scenario grid.** The plan's other checks that depend on those (that present values fall as the rate rises) are therefore not run. Nothing here is a Monte Carlo or a random draw over EPA's low and high ranges: they are scenario bounds, not distributions.

In [6]:
reproduced = bool(compare["within"].all())
print(f"reproduction achieved: {reproduced}")
print("downstream results computed in this notebook: none" if not reproduced else "downstream results: not applicable")
print(f"scenarios: {', '.join(SCENARIOS)} (EPA's own, paired); rates computed: {RATE:.0%} only")
assert not reproduced, "this notebook records a miss; if the reproduction were achieved the plan's downstream results would apply"

reproduction achieved: False
downstream results computed in this notebook: none
scenarios: low, high (EPA's own, paired); rates computed: 2% only


## Limitations

- **This is a reproduction attempt of EPA's own assumptions, not an independent estimate.** Every cost, benefit quantity, dollar value, schedule and scenario is EPA's. Nothing here estimates anything about lead in water, health, or what the rule costs.
- **A failed reproduction does not show EPA's figures are wrong.** It shows they cannot be rebuilt from the published tables under a documented convention. The cause of the gap is unknown; this notebook names one possibility (a difference in base year or flow timing) and does not test it.
- **The convention was fixed after retrieval.** The plan left the timing convention to the document; the owner's rule, committed as amendment 1 (the file E02_pap_amendment_1.md in the pap folder, SHA-256 shown in the configuration cell) after the first check had shown the gap, supplies it. The check that produced the gap therefore pre-dates the rule. The rule was applied once and no alternative was computed.
- **Departures from the plan.** The plan's central scenario (sections 4 and 6) does not exist in the document and is replaced by EPA's low and high scenarios (amendment 1). The plan's primary results at 3 and 7 percent, its scenario grid, its switching rate and its monotonicity check were not produced because the plan's stop rule for a missed reproduction holds. The plan was not edited.
- **The yearly national table is a sum of a model's system-level flows.** EPA's printed annualised values come from the model's unrounded flows, so a rebuild from the rounded yearly sum could differ for reasons unrelated to timing; the plan's tolerance allows only rounding.
- **To read Circular A-4 (2003) the text was fetched once more than the single approved document** and is quoted with its section and page; the Circular's file is the one on OMB's circulars page.
- **The rule's status was checked on one date** (31 October 2026) and court dockets were not searched. The extraction takes values from a PDF and checks them against EPA's printed totals only.

## Key Takeaways

1. **EPA's own 2 percent annualised values cannot be rebuilt from its published yearly streams under the convention applied.** All six comparisons (costs, benefits and net benefits, low and high scenarios) fall below EPA's printed values by between 1.77 and 1.98 percent, against a tolerance of 0.5 percent.
2. **The gap is close to one year of discounting.** EPA's value divided by the rebuilt value is between 1.018 and 1.020 in every comparison, beside a one-year factor of 1.020. That is consistent with a difference in base year or flow timing; EPA states no convention, so it cannot be confirmed.
3. **The tables themselves are sound.** They carry the digests the record names, every year is present for both scenarios, every net is benefits minus costs within rounding, and the undiscounted net totals ($479 billion and $918 billion) match EPA's text.
4. **A pre-set stop rule did its job.** The plan said what would happen on a miss; the miss happened; nothing was computed at 3 or 7 percent, no net present value was shown, and no alternative convention was tried to close the gap.
5. **A reproduction is a different claim from a re-analysis.** This notebook rebuilds a published number from published inputs. It makes no claim about lead, health or cost, and a miss is not a finding about EPA.

## Exercises

1. Without computing anything, say which parts of the convention an annualised value depends on and which only a present value depends on. Why does the annualised value not depend on whether flows are at the start or the end of a year?
2. List three reasons a rebuild from a rounded yearly national table could differ from a model's own annualised value, other than the base year. Which of them could produce a gap of the same proportion in costs, benefits and net benefits?
3. Write the amendment you would have asked the owner to commit if you had wanted to test a different base year, and say what it would have to state about when it was written relative to the data.

## References

U.S. Environmental Protection Agency (2024), *Economic Analysis for the Final Lead and Copper Rule Improvements*, October 2024, Exhibits ES-3, 6-4, 6-5 and 6-6, Sections 4.2.3.3 and 5.6.2. U.S. Office of Management and Budget (2003), Circular A-4, *Regulatory Analysis*, September 17, 2003, "Discount Rates" (page 32) and the section on presenting results (page 45). U.S. EPA, "Lead and Copper Rule Improvements," program page (last updated December 29, 2025), and the Federal Register notice of 30 October 2024 (2024-23549). Page and section references as recalled from the documents read on 2026-10-30; the values themselves are checked by the notebook against the committed tables.